# Integración de Datos y Cálculo del Índice Alternativo

Este notebook consolida los datos objetivo de Oliver Wyman Forum (OWF) con los indicadores de ONU y OECD explorados anteriormente, y calcula el nuevo índice de prosperidad urbana alternativa centrado en la sostenibilidad y la equidad.

In [1]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import MinMaxScaler

# Encontrar la raíz del proyecto
def obtener_raiz_proyecto():
    current = os.path.abspath(os.getcwd())
    while current != os.path.dirname(current):
        if '.git' in os.listdir(current) or 'README.md' in os.listdir(current):
            return current
        current = os.path.dirname(current)
    return os.getcwd()

def cargar_excel_seguro(path_relativo, sheet_name=0, skiprows=0):
    raiz = obtener_raiz_proyecto()
    abs_path = os.path.abspath(os.path.join(raiz, path_relativo))
    if os.name == 'nt' and len(abs_path) >= 260:
        abs_path = '\\?\\' + abs_path
    return pd.read_excel(abs_path, sheet_name=sheet_name, skiprows=skiprows)

## 1. Carga de los Datasets Procesados

In [2]:
# A. Carga de Target (Oliver Wyman)
df_owf = cargar_excel_seguro('data/raw/oliver-wyman-forum-target/book-cities-shaping-the-future.xlsx', sheet_name='Sheet1')
df_owf['City'] = df_owf['City'].astype(str).str.replace('\xa0', ' ').str.strip()

# B. Carga de Features (Cleaned en Notebook 02)
# CPI
df_cpi_raw = cargar_excel_seguro('data/raw/urban-indicator-database-features/city-prosperity-index-cpi/city_prosperity_index_database_2016.xls')
headers = df_cpi_raw.iloc[0:4].ffill(axis=1).fillna('')
new_cols = [' | '.join([str(p).replace('.0', '').strip() for p in headers.iloc[:, i] if str(p) != 'nan' and str(p).strip()]) for i in range(df_cpi_raw.shape[1])]
df_cpi = df_cpi_raw.iloc[4:].copy()
df_cpi.columns = new_cols
df_cpi = df_cpi.rename(columns={
    'City Prosperity Index. Global CPI. Base Data. 2016 | COUNTRY': 'Country',
    'City Prosperity Index. Global CPI. Base Data. 2016 | CITY': 'City'
}).dropna(subset=['City'])
df_cpi = df_cpi[df_cpi['City'] != '--']

# Gini
df_gini_raw = cargar_excel_seguro('data/raw/urban-indicator-database-features/economic-indicators/Gini_at_disposable_income_after_taxes_and_transfers,_2010_-_2017.xls')
df_gini = df_gini_raw.iloc[1:].copy()
df_gini.columns = ['Country', 'City'] + [str(int(float(y))) for y in df_gini_raw.iloc[0].tolist()[2:]]

# Áreas Verdes
df_verdes = cargar_excel_seguro('data/raw/urban-indicator-database-features/open-spaces-and-green-areas/green_areas_1990_2020.xlsx', sheet_name='Data')
df_verdes = df_verdes.rename(columns={'Country or Territory Name': 'Country', 'City Name': 'City'})

# Transporte
df_transporte = cargar_excel_seguro('data/raw/urban-indicator-database-features/urban-transport/public_transport_access.xlsx', sheet_name='Sheet1')
df_transporte = df_transporte.rename(columns={'Country or Territory Name': 'Country', 'City Name': 'City'})

print(f'Datasets cargados. OWF: {df_owf.shape[0]}, CPI: {df_cpi.shape[0]}, Gini: {df_gini.shape[0]}, Verdes: {df_verdes.shape[0]}, Transporte: {df_transporte.shape[0]}')

Datasets cargados. OWF: 1500, CPI: 333, Gini: 227, Verdes: 667, Transporte: 2353


## 2. Normalización de Nombres para Integración

Dado que los nombres de ciudades pueden contener información del país (ej: 'Seoul, Korea' en OWF) y los nombres de los países pueden variar, definiremos funciones de normalización de cadenas de caracteres.

In [3]:
def normalizar_nombre(texto):
    if not isinstance(texto, str):
        return ''
    # Limpieza estándar de espacios y caracteres
    t = texto.replace('\xa0', ' ').strip().lower()
    # Extraer la primera parte si está separada por coma (ej: 'seoul, korea' -> 'seoul')
    t = t.split(',')[0].strip()
    # Eliminar términos comunes de administración
    for word in ['city', 'metro', 'region', 'provincia', 'district', 'area']:
        t = t.replace(word, '').strip()
    return t

# Aplicar la normalización a las columnas de enlace
df_owf['City_Key'] = df_owf['City'].apply(normalizar_nombre)
df_cpi['City_Key'] = df_cpi['City'].apply(normalizar_nombre)
df_gini['City_Key'] = df_gini['City'].apply(normalizar_nombre)
df_verdes['City_Key'] = df_verdes['City'].apply(normalizar_nombre)
df_transporte['City_Key'] = df_transporte['City'].apply(normalizar_nombre)

## 3. Fusión de Datasets e Imputación de Faltantes

Realizamos el cruce de tablas evaluando la cobertura inicial de variables sobre el target.

In [4]:
# Realizar los cruces consecutivos sobre el dataframe de rankings de OWF
df_merged = df_owf.merge(df_cpi[['City_Key', 'City Prosperity Index. Global CPI. Base Data. 2016 | PRODUCTIVITY INDEX (P) | Economic Strenght | City Product per capita']], on='City_Key', how='left')
df_merged = df_merged.merge(df_gini[['City_Key', '2017']], on='City_Key', how='left')
df_merged = df_merged.merge(df_verdes[['City_Key', 'Average share of green area in city/ urban area 2020 (%)']], on='City_Key', how='left')
df_merged = df_merged.merge(df_transporte[['City_Key', 'Share of urban population with convenient access to public transport (%)']], on='City_Key', how='left')

# Renombrar columnas resultantes
df_merged = df_merged.rename(columns={
    'City Prosperity Index. Global CPI. Base Data. 2016 | PRODUCTIVITY INDEX (P) | Economic Strenght | City Product per capita': 'CPI_GDP_pc',
    '2017': 'Gini_Index',
    'Average share of green area in city/ urban area 2020 (%)': 'Green_Area_Share',
    'Share of urban population with convenient access to public transport (%)': 'Transport_Access'
})

print('Estadísticas de nulos tras el cruce:')
print(df_merged[['CPI_GDP_pc', 'Gini_Index', 'Green_Area_Share', 'Transport_Access']].isnull().sum())

KeyError: "['Average share of green area in city/urban area 2020 (%)'] not in index"

## 4. Cálculo del Índice de Prosperidad Urbana Alternativo (AUI)

Definimos un scoring ponderado:
* **Ponderación Ambiental (Green_Area_Share):** 25%
* **Ponderación de Movilidad (Transport_Access):** 25%
* **Ponderación de Equidad Social (Gini_Index):** 25% (invertido, mejor puntaje para menor Gini)
* **Ponderación Económica (Commercial Hubs/GDP):** 25%

Primero imputamos los valores nulos mediante medianas continentales/regionales.

In [ ]:
# Imputar valores nulos con la media del dataset
for col in ['CPI_GDP_pc', 'Gini_Index', 'Green_Area_Share', 'Transport_Access']:
    df_merged[col] = df_merged[col].fillna(df_merged[col].median())

# Escalar todas las variables a rango [0, 100] usando MinMaxScaler
scaler = MinMaxScaler(feature_range=(0, 100))

# Para Gini, invertimos el valor ya que menor desigualdad (menor Gini) es mejor
df_merged['Gini_Score'] = 100 - scaler.fit_transform(df_merged[['Gini_Index']])

df_merged['Green_Score'] = scaler.fit_transform(df_merged[['Green_Area_Share']])
df_merged['Transport_Score'] = scaler.fit_transform(df_merged[['Transport_Access']])

# Para el bloque económico de OWF, invertimos la jerarquía de rangos (rango 1 es el mejor, 1500 el peor)
df_merged['Economic_OWF_Score'] = 100 - scaler.fit_transform(df_merged[['Commercial Hubs']])

# Cálculo final del Alternative Urban Index (AUI) con pesos equilibrados (25% cada dimensión)
df_merged['AUI'] = (
    df_merged['Gini_Score'] * 0.25 +
    df_merged['Green_Score'] * 0.25 +
    df_merged['Transport_Score'] * 0.25 +
    df_merged['Economic_OWF_Score'] * 0.25
)

# Generar el nuevo ranking
df_merged['AUI_Rank'] = df_merged['AUI'].rank(ascending=False, method='min')

print('Top 5 ciudades según el nuevo índice alternativo (AUI):')
display(df_merged.nsmallest(5, 'AUI_Rank')[['City', 'AUI', 'AUI_Rank', 'Commercial Hubs', 'Climate Resilient']])